In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler

class MPPTDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32).unsqueeze(1) # [Batch, 1] 형태로 변경

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def preprocess_data(file_path, scaler=None):
    data = np.load(file_path)
    X, Y = data['X'], data['Y']

    # 1. Day (인덱스 4) 변환: 0 ~ 1 ~ 0
    day = X[:, :, 4]
    X[:, :, 4] = np.sin(np.pi * day / 365.0)

    # 2. Hour (인덱스 5) 변환: 1 ~ 0 ~ -1
    hour = X[:, :, 5]
    X[:, :, 5] = np.cos(np.pi * (hour - 5.0) / 14.0)

    # 3. V, I, P, dP/dV (인덱스 0~3) 스케일링
    batch, seq_len, features = X.shape
    X_reshaped = X.reshape(-1, features)

    if scaler is None:
        scaler = StandardScaler()
        X_reshaped[:, :4] = scaler.fit_transform(X_reshaped[:, :4])
    else:
        X_reshaped[:, :4] = scaler.transform(X_reshaped[:, :4])

    X_scaled = X_reshaped.reshape(batch, seq_len, features)

    return X_scaled, Y, scaler

In [3]:
class MPPTTriggerLSTM(nn.Module):
    def __init__(self, input_size=6, hidden_size=64, num_layers=2):
        super(MPPTTriggerLSTM, self).__init__()

        # LSTM 계층
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True, # 입력 형태: [Batch, Seq, Feature]
            dropout=0.4 if num_layers > 1 else 0
        )

        # 완전 연결 계층 (분류기)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32, 1) # 이진 분류이므로 출력 노드는 1개
        )

    def forward(self, x):
        # out: 모든 time step의 hidden state
        # hn: 마지막 time step의 hidden state
        out, (hn, cn) = self.lstm(x)

        # 마지막 윈도우 스텝(60번째 초)의 결과값만 사용
        last_hidden = out[:, -1, :]

        logits = self.fc(last_hidden)
        return logits # 손실 함수에서 Sigmoid를 처리하도록 설계 (BCEWithLogitsLoss)

In [4]:
# ==========================================
# [실행 환경 설정]
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 기기: {device}")

# 1. 데이터 로드 및 전처리
train_path = '/content/drive/MyDrive/mppt_dataset/mppt_thesis_dataset_merged.npz'
test_path = '/content/drive/MyDrive/mppt_dataset/mppt_thesis_dataset_1year_1.npz'

print("데이터 전처리 중...")
X_train, Y_train, scaler = preprocess_data(train_path)
X_test, Y_test, _ = preprocess_data(test_path, scaler=scaler) # 테스트 데이터는 훈련 스케일러 적용

train_dataset = MPPTDataset(X_train, Y_train)
test_dataset = MPPTDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 2. 모델, 손실 함수, 최적화 기법 설정
model = MPPTTriggerLSTM(input_size=6, hidden_size=64, num_layers=2).to(device)
criterion = nn.BCEWithLogitsLoss() # 이진 분류에 최적화된 Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. 학습 루프
num_epochs = 100

print("학습 시작...")
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

        # 정확도 계산 (Logit > 0 이면 1로 예측, 아니면 0)
        predicted = (outputs > 0).float()
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

    avg_train_loss = train_loss / total_train
    train_acc = correct_train / total_train * 100

    # 평가 루프
    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            predicted = (outputs > 0).float()
            correct_test += (predicted == labels).sum().item()
            total_test += labels.size(0)

    avg_test_loss = test_loss / total_test
    test_acc = correct_test / total_test * 100

    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
              f"Test Loss: {avg_test_loss:.4f}, Test Acc: {test_acc:.2f}%")

print("학습 완료!")
# 모델 저장
torch.save(model.state_dict(), 'mppt_lstm_model.pth')

사용 기기: cuda
데이터 전처리 중...
학습 시작...
Epoch [1/100] Train Loss: 0.3002, Train Acc: 89.58% | Test Loss: 0.1754, Test Acc: 94.62%
Epoch [5/100] Train Loss: 0.0982, Train Acc: 97.12% | Test Loss: 0.0709, Test Acc: 97.69%
Epoch [10/100] Train Loss: 0.0570, Train Acc: 98.25% | Test Loss: 0.0333, Test Acc: 99.06%
Epoch [15/100] Train Loss: 0.0364, Train Acc: 98.96% | Test Loss: 0.0232, Test Acc: 99.20%
Epoch [20/100] Train Loss: 0.0336, Train Acc: 99.05% | Test Loss: 0.0125, Test Acc: 99.58%
Epoch [25/100] Train Loss: 0.0226, Train Acc: 99.38% | Test Loss: 0.0107, Test Acc: 99.67%
Epoch [30/100] Train Loss: 0.0171, Train Acc: 99.47% | Test Loss: 0.0106, Test Acc: 99.76%
Epoch [35/100] Train Loss: 0.0304, Train Acc: 99.31% | Test Loss: 0.0118, Test Acc: 99.72%
Epoch [40/100] Train Loss: 0.0331, Train Acc: 99.19% | Test Loss: 0.0156, Test Acc: 99.62%
Epoch [45/100] Train Loss: 0.0105, Train Acc: 99.73% | Test Loss: 0.0124, Test Acc: 99.76%
Epoch [50/100] Train Loss: 0.0136, Train Acc: 99.67% | Tes

In [5]:
import torch
import numpy as np
from torch.utils.data import DataLoader

# 1. GPU 장치 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"평가에 사용할 장치: {device}")

# 2. 테스트 데이터셋 전처리 및 로드
file_path = '/content/drive/MyDrive/mppt_dataset/mppt_thesis_dataset_1year_2.npz'
# 학습 때 사용했던 scaler가 메모리에 살아있어야 합니다.

train_path = '/content/drive/MyDrive/mppt_dataset/mppt_thesis_dataset_merged.npz'
test_path = '/content/drive/MyDrive/mppt_dataset/mppt_thesis_dataset_1year_1.npz'

print("데이터 전처리 중...")
X_train, Y_train, scaler = preprocess_data(train_path)
X_test, Y_test, _ = preprocess_data(test_path, scaler=scaler) # 테스트 데이터는 훈련 스케일러 적용

train_dataset = MPPTDataset(X_train, Y_train)
test_dataset = MPPTDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

x_test_np, y_test_np, _ = preprocess_data(file_path, scaler=scaler)

# 🌟 수정 포인트: 직접 만드신 MPPTDataset 재사용 (차원 자동 맞춤 기능 활용)
test_dataset = MPPTDataset(x_test_np, y_test_np)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 3. 모델 인스턴스화 및 가중치 로드
model = MPPTTriggerLSTM(input_size=6, hidden_size=64, num_layers=2).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/mppt_dataset/mppt_lstm_model.pth', map_location=device))
model.eval()

# 4. 테스트 진행 및 정답률 계산
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device) # MPPTDataset 덕분에 [Batch, 1] 형태로 들어옴

        outputs = model(inputs)

        # 🌟 수정 포인트: 학습 루프와 동일한 이진 분류 로직 적용
        predicted = (outputs > 0).float()

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# 최종 정확도 계산
accuracy = 100 * correct / total

print(f'\n--- 평가 결과 ---')
print(f'총 데이터 수: {total}개')
print(f'맞춘 문제 수: {correct}개')
print(f'모델 정답률(Accuracy): {accuracy:.2f}%')

평가에 사용할 장치: cpu
데이터 전처리 중...

--- 평가 결과 ---
총 데이터 수: 2023개
맞춘 문제 수: 2022개
모델 정답률(Accuracy): 99.95%
